# Heart Disease - Model Training

This notebook trains multiple ML models on the heart disease dataset and saves them to the `ml_models/` folder.

In [ ]:
# Cell 1: Imports
import pandas as pd
import numpy as np
import joblib
import os
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import AdaBoostClassifier, RandomForestClassifier, GradientBoostingClassifier, VotingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.model_selection import cross_val_score

print("All imports successful")

In [ ]:
# Cell 2: Load datasets
train_df = pd.read_csv('../dataset/raw_train.csv')
val_df = pd.read_csv('../dataset/raw_val.csv')
test_df = pd.read_csv('../dataset/raw_test.csv')

print(f"Train: {train_df.shape}")
print(f"Validation: {val_df.shape}")
print(f"Test: {test_df.shape}")
train_df.head()

In [ ]:
# Cell 3: Separate features and target
feature_cols = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak', 'sex', 'cp', 'fbs', 'restecg', 'exang', 'slope', 'ca', 'thal']

X_train = train_df[feature_cols]
y_train = train_df['target']
X_val = val_df[feature_cols]
y_val = val_df['target']
X_test = test_df[feature_cols]
y_test = test_df['target']

print(f"X_train: {X_train.shape}, y_train distribution:\n{y_train.value_counts()}")
print(f"\nX_val: {X_val.shape}, y_val distribution:\n{y_val.value_counts()}")
print(f"\nX_test: {X_test.shape}, y_test distribution:\n{y_test.value_counts()}")

In [ ]:
# Cell 4: Define models
models = {
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'AdaBoost': AdaBoostClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42),
    'XGBoost': XGBClassifier(random_state=42, use_label_encoder=False, eval_metric='logloss'),
    'k-NN': KNeighborsClassifier(),
    'Naïve Bayes': GaussianNB()
}

print("Models defined:", list(models.keys()))

In [ ]:
# Cell 5: Train and evaluate each model
results = []
trained_models = {}

for name, model in models.items():
    # Train
    model.fit(X_train, y_train)
    trained_models[name] = model
    
    # Predict on validation set
    y_pred = model.predict(X_val)
    y_prob = model.predict_proba(X_val)[:, 1]
    
    # Metrics
    acc = accuracy_score(y_val, y_pred)
    prec = precision_score(y_val, y_pred, zero_division=0)
    rec = recall_score(y_val, y_pred, zero_division=0)
    f1 = f1_score(y_val, y_pred, zero_division=0)
    
    results.append({
        'Model': name,
        'Accuracy': round(acc, 4),
        'Precision': round(prec, 4),
        'Recall': round(rec, 4),
        'F1-Score': round(f1, 4)
    })
    
    print(f"{name:20s} | Acc: {acc:.4f} | Prec: {prec:.4f} | Rec: {rec:.4f} | F1: {f1:.4f}")

results_df = pd.DataFrame(results)
results_df

In [ ]:
# Cell 6: Train Ensemble (Soft Voting)
voting_clf = VotingClassifier(
    estimators=[(name, model) for name, model in trained_models.items()],
    voting='soft'
)
voting_clf.fit(X_train, y_train)
trained_models['Ensemble (Soft Voting)'] = voting_clf

y_pred_ens = voting_clf.predict(X_val)
y_prob_ens = voting_clf.predict_proba(X_val)[:, 1]

acc_ens = accuracy_score(y_val, y_pred_ens)
prec_ens = precision_score(y_val, y_pred_ens, zero_division=0)
rec_ens = recall_score(y_val, y_pred_ens, zero_division=0)
f1_ens = f1_score(y_val, y_pred_ens, zero_division=0)

results.append({
    'Model': 'Ensemble (Soft Voting)',
    'Accuracy': round(acc_ens, 4),
    'Precision': round(prec_ens, 4),
    'Recall': round(rec_ens, 4),
    'F1-Score': round(f1_ens, 4)
})

print(f"{'Ensemble (Soft Voting)':20s} | Acc: {acc_ens:.4f} | Prec: {prec_ens:.4f} | Rec: {rec_ens:.4f} | F1: {f1_ens:.4f}")

In [ ]:
# Cell 7: Save all models
os.makedirs('../ml_models', exist_ok=True)

for name, model in trained_models.items():
    safe_name = name.lower().replace(' ', '_').replace('(', '').replace(')', '')
    path = f'../ml_models/{safe_name}.pkl'
    joblib.dump(model, path)
    print(f"Saved: {path}")

# Also save the results dataframe
results_df_final = pd.DataFrame(results)
results_df_final.to_csv('../ml_models/model_performance.csv', index=False)
print("\nPerformance metrics saved to ml_models/model_performance.csv")

In [ ]:
# Cell 8: Verify saved models
print("Models in ml_models/:")
for f in os.listdir('../ml_models'):
    print(f"  - {f}")